In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. 10,000 customers ka realistic data generate karein
np.random.seed(42)
n_samples = 10000

data = {
    'CreditScore': np.random.randint(350, 850, n_samples),
    'Geography': np.random.choice(['France', 'Spain', 'Germany'], n_samples, p=[0.5, 0.25, 0.25]),
    'Gender': np.random.choice(['Female', 'Male'], n_samples, p=[0.45, 0.55]),
    'Age': np.random.randint(18, 85, n_samples),
    'Tenure': np.random.randint(0, 11, n_samples),
    'Balance': np.where(np.random.rand(n_samples) > 0.3, np.random.uniform(20000, 200000, n_samples), 0), # 30% logon ka balance 0 hoga
    'NumOfProducts': np.random.choice([1, 2, 3, 4], n_samples, p=[0.5, 0.4, 0.08, 0.02]),
    'HasCrCard': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
    'IsActiveMember': np.random.choice([0, 1], n_samples, p=[0.5, 0.5]),
    'EstimatedSalary': np.random.uniform(10000, 200000, n_samples)
}

# Churn (Exited) logic: Age zyada ho, Balance high ho, aur active member na ho toh churn ke chances zyada honge (Realistic Behavior)
log_odds = (data['Age'] - 40) * 0.05 + (data['Balance'] / 100000) * 0.2 - data['IsActiveMember'] * 0.8 - 1.5
probabilities = 1 / (1 + np.exp(-log_odds))
data['Exited'] = np.where(np.random.rand(n_samples) < probabilities, 1, 0)

df = pd.DataFrame(data)

print("--- 10,000 Rows Dataset Successfully Generated ---")
display(df.head())

print(f"\nDataset Shape: {df.shape} (Ab Deep Learning ke liye perfect hai!)")
print("\nTarget Class Distribution (Exited):")
print(df['Exited'].value_counts())

--- 10,000 Rows Dataset Successfully Generated ---


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,452,France,Male,75,3,140447.947224,1,0,1,46451.697069,1
1,785,France,Female,70,5,150704.530468,2,0,1,76389.527066,1
2,698,France,Male,67,9,192757.275426,2,1,1,20056.557113,1
3,620,Spain,Male,21,6,172989.320718,3,1,0,20552.541723,0
4,456,France,Male,69,6,167969.252697,2,1,1,61822.905445,1



Dataset Shape: (10000, 11) (Ab Deep Learning ke liye perfect hai!)

Target Class Distribution (Exited):
Exited
0    7202
1    2798
Name: count, dtype: int64


In [5]:
# Categorical mapping
le_geo = LabelEncoder()
df['Geography'] = le_geo.fit_transform(df['Geography'])

le_gender = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender'])

X = df.drop(columns=['Exited']).values
y = df['Exited'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Preprocessing Done on 10k rows!")

Preprocessing Done on 10k rows!


In [6]:
from torch.utils.data import DataLoader, TensorDataset

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("DataLoader Ready!")

DataLoader Ready!


In [7]:
class ChurnANN(nn.Module):
    def __init__(self, input_dim):
        super(ChurnANN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(16, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.relu1(self.fc1(x))
        out = self.relu2(self.fc2(out))
        out = self.sigmoid(self.fc3(out))
        return out

input_dim = X_train.shape[1]
model = ChurnANN(input_dim)
print(model)

ChurnANN(
  (fc1): Linear(in_features=10, out_features=16, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=8, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [10]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005) # Ir badhaya taake jaldi seekhe

epochs = 100

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] -> Loss: {epoch_loss/len(train_loader):.4f}")

Epoch [1/100] -> Loss: 0.4849
Epoch [5/100] -> Loss: 0.4846
Epoch [10/100] -> Loss: 0.4824
Epoch [15/100] -> Loss: 0.4822
Epoch [20/100] -> Loss: 0.4800
Epoch [25/100] -> Loss: 0.4806
Epoch [30/100] -> Loss: 0.4796
Epoch [35/100] -> Loss: 0.4785
Epoch [40/100] -> Loss: 0.4784
Epoch [45/100] -> Loss: 0.4783
Epoch [50/100] -> Loss: 0.4765
Epoch [55/100] -> Loss: 0.4762
Epoch [60/100] -> Loss: 0.4761
Epoch [65/100] -> Loss: 0.4750
Epoch [70/100] -> Loss: 0.4750
Epoch [75/100] -> Loss: 0.4746
Epoch [80/100] -> Loss: 0.4746
Epoch [85/100] -> Loss: 0.4737
Epoch [90/100] -> Loss: 0.4734
Epoch [95/100] -> Loss: 0.4737
Epoch [100/100] -> Loss: 0.4735


In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    predicted_classes = (test_preds >= 0.5).float()
    correct = (predicted_classes == y_test_t).sum().item()
    accuracy = (correct / y_test_t.size(0)) * 100

print(f"🔥 Final Model Accuracy on 10k Dataset: {accuracy:.2f}%")

🔥 Final Model Accuracy on 10k Dataset: 72.85%


: 